# SPY Risk Alert Project Pipeline

**Stage 04 - Data Acquisition and Ingestion**

This cumulative pipeline starts with a documented Nasdaq SPY daily OHLCV acquisition. It saves a timestamped raw snapshot and checksum manifest. The provider data are unadjusted; this notebook does not construct a return target or make a trading recommendation.

## 1. Project Root and Imports

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')  # project/notebooks -> project
ROOT = Path.cwd()
if not (ROOT / 'src' / 'ingestion.py').is_file():
    for candidate in (ROOT, *ROOT.parents):
        project_candidate = candidate / 'project'
        if (project_candidate / 'src' / 'ingestion.py').is_file():
            ROOT = project_candidate
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working from:', ROOT.name)

from datetime import date

from src.ingestion import (
    fetch_nasdaq_history,
    timestamp_utc,
    validate_spy_history,
    write_manifest,
    write_raw_csv,
)

RAW_DIR = ROOT / 'data' / 'raw'

working from: project


## 2. Request, Parse, and Validate

The Nasdaq endpoint needs no API key for this request. Query parameters, the unadjusted price convention, and all validation results are saved in the manifest.

In [2]:
SYMBOL = 'SPY'
END_DATE = date.today()
START_DATE = END_DATE.replace(year=END_DATE.year - 10)

spy_raw, source_metadata = fetch_nasdaq_history(
    SYMBOL, start_date=START_DATE.isoformat(), end_date=END_DATE.isoformat()
)
validation = validate_spy_history(spy_raw)

print('Rows and columns:', validation['shape'])
print('Date range:', validation['date_min'], 'to', validation['date_max'])
print('Price convention:', source_metadata['price_convention'])
spy_raw.head()

Rows and columns: [2512, 6]
Date range: 2016-09-07 to 2026-09-04
Price convention: unadjusted OHLCV as supplied by the endpoint


,date,open,high,low,close,volume
0,2016-09-07,218.84,219.22,218.3,219.01,76302150
1,2016-09-08,218.62,218.94,218.15,218.51,73855230
2,2016-09-09,216.97,217.03,213.25,213.28,220309300
3,2016-09-12,212.39,216.81,212.31,216.34,167653400
4,2016-09-13,214.84,215.1499,212.5,213.23,182323200


## 3. Save an Immutable-Style Raw Snapshot and Manifest

In [3]:
snapshot_timestamp = timestamp_utc()
raw_path = write_raw_csv(
    spy_raw, RAW_DIR, 'api_nasdaq_spy_daily', timestamp=snapshot_timestamp
)
manifest_path = write_manifest(
    {
        'path': raw_path,
        'dataset': 'SPY daily unadjusted OHLCV',
        'rows': len(spy_raw),
        'columns': list(spy_raw.columns),
        'source_metadata': source_metadata,
        'validation': validation,
    },
    RAW_DIR / f'ingestion_manifest_{snapshot_timestamp}.json',
)

print('Saved raw snapshot:', raw_path.name)
print('Saved manifest:', manifest_path.name)

Saved raw snapshot: api_nasdaq_spy_daily_20260907-143336.csv
Saved manifest: ingestion_manifest_20260907-143336.json


## Sources, Assumptions, and Risks

- Source documentation and validation rules are in `docs/data_sources.md`.
- This pipeline stores raw unadjusted OHLCV; no adjusted-close or event-label decision is made here.
- The provider can change schema, availability, or access conditions. Validation failures should stop the pipeline rather than generate misleading data.
- Future stages will extend this same notebook with preprocessing, EDA, features, modeling, and reporting.